# 04b — Driver Manifestations (Morphological Box)

For each unified technology driver, determine **2–4 domain-specific manifestations** (Ausprägungen) — concrete, mutually exclusive states this driver could reach by 2035.

These manifestations form the **Zwicky Box** (morphological box) that will be combined into scenario skeletons in NB05b.

- Input: `merge_state.json` (14 unified drivers)
- Output: `morphbox_state.json` (drivers × manifestations)

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from src.llm import safe_chat_json
from src.rag import get_collection, retrieve, format_rag_chunks
from src.models.drivers import TechDriver
from src.models.morphological import DriverManifestation, MorphologicalBox
from src import prompts

with open("../data/outputs/merge_state.json") as f:
    merge_state = json.load(f)
drivers = [TechDriver(**d) for d in merge_state["unified_drivers"]]

collection = get_collection()
print(f"Loaded {len(drivers)} drivers, KB collection: {collection.count()} chunks")

Loaded 40 drivers, KB collection: 2850 chunks


In [2]:
all_manifestations: list[DriverManifestation] = []
manifestation_map: dict[str, list[str]] = {}

for i, driver in enumerate(drivers):
    print(f"\n[{i+1}/{len(drivers)}] {driver.name[:60]}")

    chunks = retrieve(collection, f"{driver.name} {driver.description[:100]}", n=5)
    rag_text = format_rag_chunks(chunks)

    prompt = prompts.MANIFESTATION_DETERMINE.format(
        driver_name=driver.name,
        driver_description=driver.description,
        driver_origin=driver.origin.value,
        driver_confidence=driver.confidence.value,
        rag_chunks=rag_text,
    )

    result = safe_chat_json(prompt, system="You are determining technology manifestations for regulatory frequency monitoring foresight. Be specific and domain-grounded.")

    manif_ids = []
    for m in result.get("manifestations", []):
        dm = DriverManifestation(
            driver_id=driver.id,
            label=m["label"],
            description=m["description"],
            plausibility=m.get("plausibility", "medium"),
            source_chunk_ids=result.get("source_chunk_ids_used", []),
        )
        all_manifestations.append(dm)
        manif_ids.append(dm.id)
        print(f"  [{dm.plausibility:6s}] {dm.label}")

    manifestation_map[driver.id] = manif_ids

print(f"\n{'='*60}")
print(f"Total: {len(all_manifestations)} manifestations for {len(drivers)} drivers")
print(f"Average: {len(all_manifestations)/len(drivers):.1f} per driver")


[1/40] Advanced Geolocation Techniques Using Distributed Sensors (T


  [high  ] Dense urban sensor mesh with centimeter-level accuracy
  [medium] Regional networks with meter-level accuracy and adaptive sensor deployment
  [medium] Limited urban coverage with decameter-level accuracy due to multipath and synchronization limits
  [low   ] Fragmented and inconsistent geolocation due to infrastructure and synchronization failures

[2/40] High-Bandwidth I/Q Data Streaming Interfaces and Edge Comput


  [high  ] Integrated AI-Driven Edge Processing with 2 GHz I/Q Streaming
  [medium] Modular Wideband I/Q Streaming with Partial Edge Analytics
  [medium] Limited Bandwidth I/Q Streaming with Centralized Processing
  [low   ] Stagnant I/Q Interface Bandwidth and Minimal Edge Computing

[3/40] Real-Time FPGA/ASIC Pipeline and Edge Computing for Spectrum


  [high  ] Fully Parallel Real-Time Edge FPGA/ASIC Pipelines
  [medium] Hybrid Edge-Cloud Spectrum Sensing with Adaptive Pipelines
  [medium] Incremental FPGA/ASIC Pipeline Improvements with Residual Blind Times
  [low   ] Stagnation Due to Power and Complexity Constraints

[4/40] FFT Processing Engine and Cognitive Radio Spectrum Sensing T


  [high  ] Ultra-high-rate real-time FFT with continuous blind-time-free sensing
  [high  ] Edge-embedded cognitive spectrum sensing with multi-parameter FFT analytics
  [medium] Incremental FFT speed improvements with partial blind-time gaps
  [low   ] Limited FFT processing constrained by hardware and algorithmic bottlenecks

[5/40] Real-Time Spectrum Path and Cognitive Radio Spectrum Sensing


...

  [medium] Seamless Real-Time Capture with AI-Driven Offline Analysis
  [high  ] Hybrid Real-Time and Selective Offline Processing with Triggered Storage
  [medium] Continuous Real-Time Analysis with Limited Offline Storage
  [low   ] Real-Time Analysis Without Integrated Data Recording

[37/40] Big Data Architectures for Spectrum Data Management


  [medium] Fully Integrated AI-Driven Big Data Spectrum Platform
  [high  ] Hybrid Centralized-Distributed Big Data Monitoring with Human Oversight
  [medium] Fragmented Big Data Systems with Limited Interoperability
  [low   ] Minimal Big Data Adoption with Legacy Systems Dominant

[38/40] Open Spectrum Data as a Service with APIs


  [medium] Global interoperable open spectrum data ecosystem
  [high  ] Regional open spectrum data hubs with limited interoperability
  [medium] Fragmented, proprietary spectrum data silos dominate
  [low   ] Open spectrum data services collapse due to privacy and data overload

[39/40] Enhanced Data Security and Privacy Measures Including Blockc


  [medium] Blockchain-Enabled Transparent Spectrum Ledger
  [high  ] Edge AI with Federated Learning Secures Distributed Monitoring
  [high  ] Standardized Encryption and Authentication with RF Fingerprinting
  [medium] Fragmented Security with Limited Blockchain and Privacy Adoption

[40/40] Integration of Mobile and Portable Spectrum Monitoring Syste


  [medium] Fully Integrated Mobile-Portable Hybrid Networks
  [high  ] Modular Portable Systems with On-Demand Deployment
  [medium] Mobile-Portable Systems Limited by Antenna and Deployment Constraints
  [low   ] Minimal Integration and Reliance on Fixed Stations

Total: 159 manifestations for 40 drivers
Average: 4.0 per driver


## Validation: Embedding Similarity Check

Flag manifestations of the same driver that are too similar (cosine > 0.9) — they may not be truly mutually exclusive.

In [3]:
import numpy as np
from src.llm import embed

manif_lookup = {m.id: m for m in all_manifestations}
warnings = 0

for driver in drivers:
    m_ids = manifestation_map[driver.id]
    if len(m_ids) < 2:
        print(f"⚠ {driver.name[:40]}: only {len(m_ids)} manifestation(s)")
        warnings += 1
        continue

    texts = [f"{manif_lookup[mid].label}: {manif_lookup[mid].description}" for mid in m_ids]
    embs = np.array(embed(texts))
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    sim = (embs @ embs.T) / (norms @ norms.T + 1e-10)

    for a in range(len(m_ids)):
        for b in range(a + 1, len(m_ids)):
            if sim[a][b] > 0.90:
                print(f"⚠ {driver.name[:40]}: '{manif_lookup[m_ids[a]].label}' ↔ '{manif_lookup[m_ids[b]].label}' similarity={sim[a][b]:.3f}")
                warnings += 1

if warnings == 0:
    print("✓ All manifestations pass similarity check")

✓ All manifestations pass similarity check


## Morphological Box (Zwicky Box)

In [4]:
driver_lookup = {d.id: d for d in drivers}

print(f"{'DRIVER':<45} | MANIFESTATIONS (optimistic → pessimistic)")
print("=" * 120)
for driver in drivers:
    m_ids = manifestation_map[driver.id]
    labels = [manif_lookup[mid].label for mid in m_ids]
    plaus = [manif_lookup[mid].plausibility for mid in m_ids]
    row = " | ".join([f"{l} [{p}]" for l, p in zip(labels, plaus)])
    print(f"{driver.name[:44]:<45} | {row}")

DRIVER                                        | MANIFESTATIONS (optimistic → pessimistic)
Advanced Geolocation Techniques Using Distri  | Dense urban sensor mesh with centimeter-level accuracy [high] | Regional networks with meter-level accuracy and adaptive sensor deployment [medium] | Limited urban coverage with decameter-level accuracy due to multipath and synchronization limits [medium] | Fragmented and inconsistent geolocation due to infrastructure and synchronization failures [low]
High-Bandwidth I/Q Data Streaming Interfaces  | Integrated AI-Driven Edge Processing with 2 GHz I/Q Streaming [high] | Modular Wideband I/Q Streaming with Partial Edge Analytics [medium] | Limited Bandwidth I/Q Streaming with Centralized Processing [medium] | Stagnant I/Q Interface Bandwidth and Minimal Edge Computing [low]
Real-Time FPGA/ASIC Pipeline and Edge Comput  | Fully Parallel Real-Time Edge FPGA/ASIC Pipelines [high] | Hybrid Edge-Cloud Spectrum Sensing with Adaptive Pipelines [medium] | Incr

In [5]:
morph_box = MorphologicalBox(
    drivers=[d.id for d in drivers],
    manifestations=manifestation_map,
    all_manifestations=all_manifestations,
)

morphbox_state = {
    "drivers": morph_box.drivers,
    "manifestations": morph_box.manifestations,
    "all_manifestations": [m.model_dump(mode="json") for m in morph_box.all_manifestations],
}

with open("../data/outputs/morphbox_state.json", "w") as f:
    json.dump(morphbox_state, f, indent=2)

print(f"Saved morphological box: {len(morph_box.drivers)} drivers × {len(morph_box.all_manifestations)} manifestations")

Saved morphological box: 40 drivers × 159 manifestations
